# Acervo que Fala — Notebook 02: alt-text para os 5 objetos do smoke test

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

No Notebook 01, um objeto atravessou a **observação visual** sem inventar nada. Este notebook dá o próximo passo do pipeline: a **redação do alt-text (nível 1)** — e roda o caminho completo nos **5 objetos do smoke test** do projeto, comparando o resultado com os alt-texts escritos à mão na proposta (o gabarito humano).

O pipeline em duas etapas separadas, de propósito:

1. **Observação** — o modelo vê SÓ a fotografia e descreve o visível;
2. **Redação** — o modelo (sem ver a imagem) recebe a observação + os dados do museu e escreve o alt-text seguindo as regras do projeto.

Separar as duas permite auditar cada uma: se o alt-text sair errado, sabemos se o erro nasceu no olho ou na escrita.

*Metodologia: projeto construído por um designer com LLMs como suporte (vibe coding) — cada célula explicada, como nos demais notebooks.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4**
2. **Ambiente de execução → Executar tudo**
3. Tempo total: **~15 minutos** (download do modelo + 5 objetos × 2 etapas)

## Etapa 1 — Instalar as ferramentas (~2 min)

Mesma instalação do Notebook 01, com a mesma regra aprendida lá: **a Pillow fica travada na versão que o Colab já usa** (atualizá-la quebra o ambiente — problema conhecido da comunidade).

In [ ]:
import PIL
!pip install -q -U transformers accelerate bitsandbytes pillow=={PIL.__version__}
print(f"ferramentas instaladas ✓ (Pillow mantida em {PIL.__version__})")

### Checagem do ambiente (5 segundos)

Testa os imports sensíveis a conflito de versão e confirma a GPU — antes de qualquer download grande. Se falhar: **Ambiente de execução → Desconectar e excluir ambiente de execução** e rode tudo de novo.

In [ ]:
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image

print(f"transformers {transformers.__version__} | torch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU ativa ✓ ({torch.cuda.get_device_name(0)})")
else:
    print("⚠ GPU NÃO está ativa — menu Ambiente de execução → Alterar o tipo → T4 GPU")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 5 objetos do smoke test

Os mesmos 5 objetos usados desde a proposta, um de cada categoria: o **Pote Karajá** (cerâmica), a **Faixa frontal Kalapalo** (plumária), a **Flauta de pã Tukano** (instrumento), o **Abano Fulni-ô** (trançado — a foto é um close da trama, caso difícil de propósito) e a **Tanga Tiriyó** (miçangaria). A célula baixa foto e metadados de cada um pela API pública do museu.

In [ ]:
import io, re, requests
from PIL import Image

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS = [9196, 665, 51023, 63283, 78838]

objetos = []
for item_id in IDS:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    metadados = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto, "metadados": metadados})
    print(f"✓ {item_id} — {item['title']} ({metadados.get('Povo', '?')})")
print(f"{len(objetos)} objetos carregados")

## Etapa 3 — Carregar o modelo (~5 min na primeira vez)

Mesmo modelo do Notebook 01: **Qwen3-VL-8B-Instruct**, quantizado em 4-bit para caber na GPU gratuita. Um detalhe de arquitetura: este modelo faz as **duas** etapas do pipeline — na observação ele usa a visão; na redação, só a parte de linguagem (sem receber a imagem). Um modelo em vez de dois é o que cabe na memória da GPU gratuita, e a separação das etapas continua valendo porque o que muda é **o que ele recebe** em cada uma.

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    """Envia uma conversa ao modelo e devolve só o texto novo gerado."""
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("modelo carregado ✓")

## Etapa 4 — Observação visual, prompt v2 (~5 min)

O prompt de observação foi **ajustado com base no que o Notebook 01 revelou**. Lá, o modelo não inventou nada (ótimo), mas deixou de citar três coisas que estavam na foto: a inclinação do objeto, a boca/interior visíveis e a faixa escura da borda. A versão 2 pede explicitamente:

- a **posição/orientação** do objeto (de pé, inclinado, deitado) e **partes internas visíveis**;
- as **cores de bordas, faixas e acabamentos**, não só as dominantes.

Esse ciclo — rodar, conferir com olhos humanos, ajustar a instrução — é o método do projeto inteiro.

In [ ]:
PROMPT_OBSERVACAO_V2 = (
    "Descreva APENAS o que está visível nesta fotografia de um objeto de museu:\n"
    "- formas, cores e materiais aparentes — incluindo cores de bordas, faixas e acabamentos, não só as dominantes;\n"
    "- a posição/orientação do objeto (de pé, inclinado, deitado) e se partes internas (boca, interior, verso) estão visíveis;\n"
    "- o enquadramento: o objeto aparece inteiro ou só um detalhe/close?;\n"
    "- o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).\n"
    "NÃO invente o que não dá para ver. Se algo estiver ilegível ou incerto, diga isso em vez de estimar. "
    "Responda em português."
)

for obj in objetos:
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V2}]
    )
    print(f"— {obj['titulo']} ({obj['id']}):\n{obj['observacao']}\n")

## Etapa 5 — Redação do alt-text (~3 min)

Agora o modelo **não vê a imagem**: recebe a observação da etapa anterior + três dados do registro do museu (nome, povo, categoria) e escreve o alt-text seguindo as regras do projeto:

- uma frase, **até 30 palavras**, objeto primeiro;
- linguagem simples, sem jargão de catalogação;
- descreve **a fotografia** (se for detalhe/close, avisa);
- só usa do registro o nome do objeto e o povo — todo o resto tem que vir da observação;
- responde em **JSON** (formato estruturado que o programa consegue conferir automaticamente).

Pedir JSON não é capricho: é o que permite validar cada resposta por código na avaliação dos 40 casos.

In [ ]:
import json

def extrair_json(texto):
    """O modelo às vezes envolve o JSON em ```; esta função limpa e interpreta."""
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

MODELO_PROMPT_REDACAO = """Você escreve alt-text (texto alternativo) de acessibilidade para o acervo digital de um museu, lido por pessoas cegas via leitor de tela.

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (única fonte do que é visível):
{observacao}

DADOS DO REGISTRO DO MUSEU (use SOMENTE nome e povo):
Nome do item: {nome} | Povo: {povo} | Categoria: {categoria}

REGRAS:
1. Uma frase, no máximo 30 palavras, começando pelo objeto (ex.: "Pote de cerâmica Karajá...").
2. Linguagem simples — nada de jargão de catalogação.
3. Descreva A FOTOGRAFIA: inclua orientação/enquadramento; se for detalhe/close, comece avisando isso.
4. Não afirme nada que não esteja na observação (exceto nome do objeto e povo).
5. Responda APENAS com JSON neste formato: {{"alt_text": "..."}}"""

for obj in objetos:
    prompt = MODELO_PROMPT_REDACAO.format(
        observacao=obj["observacao"],
        nome=obj["metadados"].get("Nome do item", obj["titulo"]),
        povo=obj["metadados"].get("Povo", ""),
        categoria=obj["metadados"].get("Categoria", ""),
    )
    resposta = gerar([{"type": "text", "text": prompt}], max_tokens=200)
    try:
        obj["alt_text"] = extrair_json(resposta)["alt_text"]
        obj["json_valido"] = True
    except Exception as e:
        obj["alt_text"] = resposta
        obj["json_valido"] = False
    print(f"— {obj['titulo']}: {obj['alt_text']}\n")

## Etapa 6 — Comparação com o gabarito humano

Os alt-texts escritos à mão na proposta do projeto são a **referência de qualidade** (não a resposta única certa — descrição admite variação). A tabela abaixo põe lado a lado o gerado e o manual, com duas checagens automáticas: o JSON veio válido? e o alt-text respeitou o limite de 30 palavras?

In [ ]:
GABARITO = {
    9196: "Pequeno pote de cerâmica Karajá em tom cru, decorado com grafismos geométricos em vermelho e preto, fotografado inclinado, mostrando a boca de borda escura.",
    # 665: gabarito corrigido em 24/08/2026 — o modelo detectou uma segunda pena azul
    # (parcialmente encoberta) que o gabarito original não registrava; confirmada por
    # revisão humana na foto. Registro do episódio em docs/ETAPAS.md (E5).
    665: "Faixa frontal emplumada Kalapalo: base de penas amarelas com faixa preta na borda, de onde se erguem penas longas — duas alaranjadas, uma rajada em preto e branco e duas azuis.",
    51023: "Flauta de pã Tukano com seis tubos de taquara em comprimentos decrescentes, presos por amarrações de fibra trançada, fotografada sobre fundo preto.",
    63283: "Detalhe aproximado da trama de um abano Fulni-ô: fitas de fibra vegetal entrelaçadas em padrão de espinha-de-peixe, em tons dourados e castanhos.",
    78838: "Tanga retangular de miçangas Tiriyó, chamada keweyu, tecida em branco com grafismos pretos — gregas, motivos piramidais e figuras estilizadas — e franjas nas bordas.",
}

print("=" * 100)
for obj in objetos:
    palavras = len(obj["alt_text"].split())
    print(f"\n### {obj['titulo']} ({obj['id']}) — JSON válido: {obj['json_valido']} | {palavras} palavras {'✓' if palavras <= 30 else '⚠ acima de 30'}")
    print(f"GERADO:  {obj['alt_text']}")
    print(f"MANUAL:  {GABARITO[obj['id']]}")
print("\n" + "=" * 100)
validos = sum(1 for o in objetos if o["json_valido"])
print(f"Resumo: {validos}/5 JSONs válidos | {sum(1 for o in objetos if len(o['alt_text'].split()) <= 30)}/5 dentro de 30 palavras")

## Etapa 7 — Salvar no Drive

Mesmo procedimento dos demais notebooks: o resultado completo (observações, alt-texts, checagens) vai para `Projeto_LLM/resultados/`, com nome padronizado. O Colab vai pedir autorização para conectar ao seu Drive.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM/resultados"
os.makedirs(PASTA, exist_ok=True)

resultado = {
    "notebook": "02_nivel1_smoke_test_v1",
    "modelo": MODELO,
    "prompt_observacao": PROMPT_OBSERVACAO_V2,
    "prompt_redacao": MODELO_PROMPT_REDACAO,
    "itens": [
        {
            "id": o["id"], "titulo": o["titulo"],
            "observacao": o["observacao"], "alt_text": o["alt_text"],
            "json_valido": o["json_valido"], "gabarito": GABARITO[o["id"]],
        }
        for o in objetos
    ],
}
destino = os.path.join(PASTA, "02_nivel1_smoke_test.json")
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude no chat que o Notebook 02 terminou — ele busca o resultado direto no Drive e faz a análise caso a caso (o caso mais importante é o **Abano**: o alt-text gerado avisou que a foto é um close?).

**O que este notebook provou:** o pipeline de nível 1 completo (observação → redação com regras → JSON verificável) nos 5 objetos, com comparação contra referência humana. **O que ainda não provou:** o nível 2 (descrição do objeto com o registro completo e RAG), as flags de divergência, e a escala dos 40 casos — próximos notebooks.